In [7]:
import pandas as pd
import random

# --- Configuration ---
DATASET_SIZE = 4000

# --- Chemical Knowledge Bank (SMILES & Properties) ---
# Substrates: (SMILES, Class, Leaving Group)
substrates = [
    # Methyl
    ("CBr", "methyl", "Br"), ("CCl", "methyl", "Cl"), ("CI", "methyl", "I"),
    # Primary (1°)
    ("CCBr", "1°", "Br"), ("CCCCl", "1°", "Cl"), ("CC(C)CBr", "1°", "Br"), ("CCCCI", "1°", "I"),
    # Secondary (2°)
    ("CC(Br)C", "2°", "Br"), ("CCC(Cl)C", "2°", "Cl"), ("CC(I)C", "2°", "I"), ("C1CCCCC1Br", "2°", "Br"),
    # Tertiary (3°)
    ("CC(C)(C)Br", "3°", "Br"), ("CC(C)(C)Cl", "3°", "Cl"), ("CCC(C)(I)C", "3°", "I"), ("CC(C)(Br)CC", "3°", "Br"),
    # Alkenes (For Electrophilic Addition)
    ("C=C", "alkene", "None"), ("CC=C", "alkene", "None"), ("CC(C)=C", "alkene", "None"),
    # Alkanes (For Radical Substitution)
    ("CCC", "alkane", "None"), ("CC(C)C", "alkane", "None"), ("C1CCCCC1", "alkane", "None")
]

# Reagents: (SMILES, Role)
reagents = [
    # Strong Nuc / Strong Base
    ("[OH-]", "strong_base"), ("CC[O-]", "strong_base"), # Can act as both, logic handles context
    # Strong Nuc / Weak Base
    ("I-", "strong_nucleophile"), ("CN-", "strong_nucleophile"), ("RS-", "strong_nucleophile"),
    # Weak Nuc / Weak Base
    ("O", "weak_nucleophile"), ("CO", "weak_nucleophile"), # H2O, Methanol
    # Radical Sources
    ("Br2", "radical_source"), ("Cl2", "radical_source"),
    # Acids (Electrophiles)
    ("Br", "acid"), ("Cl", "acid") # Representing HBr/HCl simplified
]

solvents = [
    ("O", "polar_protic"), ("CO", "polar_protic"), ("CC(=O)O", "polar_protic"), # Water, MeOH, Acetic Acid
    ("CN(C)C=O", "polar_aprotic"), ("CS(=O)C", "polar_aprotic"), ("CC(=O)C", "polar_aprotic"), # DMF, DMSO, Acetone
    ("C1CCCCC1", "non_polar"), ("ClC(Cl)Cl", "non_polar") # Cyclohexane, Chloroform
]

temperatures = ["low", "room_temp", "heat"]

# --- Mechanism Logic Engine ---
def determine_mechanism(sub_type, lg, r_role, solv_type, temp):
    """
    Applies Class 11/12th Organic Chem rules to determine mechanism.
    """

    # 1. Radical Substitution (Alkane + Radical Source + UV/Heat)
    if sub_type == "alkane":
        if r_role == "radical_source" and (temp == "heat" or temp == "room_temp"): # Assuming UV implied for room_temp in data context
            return "radical_substitution"
        return "no_reaction"

    # 2. Electrophilic Addition (Alkene + Acid/Electrophile)
    if sub_type == "alkene":
        if r_role == "acid" or r_role == "radical_source": # Br2 adds to alkenes too
            return "electrophilic_addition"
        return "no_reaction"

    # 3. Alkyl Halide Mechanisms (SN1/SN2/E1/E2)
    if lg in ["Br", "Cl", "I"]:

        # Methyl: Almost always SN2 if nuc present
        if sub_type == "methyl":
            if "nucleophile" in r_role or r_role == "strong_base":
                return "SN2"

        # Tertiary (3°): Steric hindrance blocks SN2
        elif sub_type == "3°":
            if r_role == "strong_base" and temp == "heat":
                return "E2"
            elif r_role == "weak_nucleophile" and solv_type == "polar_protic":
                return "E1" if temp == "heat" else "SN1"
            elif r_role == "strong_base":
                return "E2"

        # Primary (1°): Unhindered, favors SN2
        elif sub_type == "1°":
            if r_role == "strong_base" and "bulky" in r_role: # Simplified check
                return "E2"
            if "nucleophile" in r_role or r_role == "strong_base":
                return "SN2"

        # Secondary (2°): The tricky middle ground
        elif sub_type == "2°":
            if r_role == "strong_base" and temp == "heat":
                return "E2"
            elif r_role == "strong_nucleophile" and solv_type == "polar_aprotic":
                return "SN2"
            elif r_role == "weak_nucleophile" and solv_type == "polar_protic":
                return "SN1"

    return "other" # Or no_reaction

# --- Generator Loop ---
data = []
for i in range(1, DATASET_SIZE + 1):
    # Randomly select components
    sub = random.choice(substrates)
    rgt = random.choice(reagents)
    solv = random.choice(solvents)
    temp = random.choice(temperatures)

    # Apply Logic
    mech = determine_mechanism(sub[1], sub[2], rgt[1], solv[1], temp)

    # If "other" or "no_reaction", try again to ensure quality data (optional, but good for density)
    # For this script, we will keep them to show negative cases, or filter them.
    # Let's simple keep non-trivial cases mostly:
    while mech == "no_reaction":
        sub = random.choice(substrates)
        rgt = random.choice(reagents)
        mech = determine_mechanism(sub[1], sub[2], rgt[1], solv[1], temp)

    row = {
        "reaction_id": i,
        "substrate_smiles": sub[0],
        "substrate_class": sub[1],
        "leaving_group": sub[2],
        "reagent_smiles": rgt[0],
        "reagent_role": rgt[1],
        "solvent_type": solv[1],
        "temperature_class": temp,
        "mechanism_class": mech
    }
    data.append(row)

# --- Save ---
df = pd.DataFrame(data)
# df.to_csv("organic_chemistry_mechanisms.csv", index=False)
print(df.head(10))
print(f"\nGenerated {len(df)} rows.")

   reaction_id substrate_smiles substrate_class leaving_group reagent_smiles  \
0            1       CCC(C)(I)C              3°             I            Cl2   
1            2            CCCCI              1°             I             Br   
2            3         CCC(Cl)C              2°            Cl             Br   
3            4            CCCCl              1°            Cl             CO   
4            5         CCC(Cl)C              2°            Cl            Br2   
5            6            CCCCI              1°             I            RS-   
6            7            CCCCI              1°             I          [OH-]   
7            8               CI          methyl             I            CN-   
8            9       CCC(C)(I)C              3°             I             CO   
9           10               CI          methyl             I         CC[O-]   

         reagent_role   solvent_type temperature_class mechanism_class  
0      radical_source  polar_aprotic         r

In [ ]:
%pip install rdkit
from rdkit import Chem

def canonicalize(smiles):
    mol = Chem.MolFromSmiles(smiles)
    return Chem.MolToSmiles(mol)

df["substrate_smiles"] = df["substrate_smiles"].apply(canonicalize)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 36.6/36.6 MB 18.7 MB/s eta 0:00:00


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>